# Bước A10 — Kiểm chứng chéo `A8 Champion` (`DINOv3 Auto-ROI Fusion` + `Imbalanced-DYDM`) trên `NEU-DET`
- **Nhánh:** `duytung/14-neudet-dinov3-autoroi`
- **Mục tiêu:** Chứng minh phương pháp vô địch `A8` tổng quát hóa sang bộ lỗi bề mặt thép công nghiệp thứ 2 **`NEU-DET`** (`6` lớp lỗi: 3 lớp cũ `S0` + 3 lớp mới không nhãn `S1`, `b3t1`, 0 XML, 0 YOLO).
- **Tự động so sánh:** `DINOv3 Full (N/NL)` vs `DINOv3 Auto-ROI Fusion (N/NL)` vs `Imbalanced-DYDM (A10_Imb_head, km, blend50, blend70)` qua 5 seed `s0..s4`.
- **Thời gian dự kiến (GPU L4):** ~20–25 phút trọn bộ, hỗ trợ auto-resume từng bước trên Google Drive.

In [ ]:
# Cell 1: Kết nối Google Drive & kiểm tra GPU
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi
!ls -la /content/drive/MyDrive/Datasets/NEU-DET || echo '⚠️ Kiểm tra lại vị trí thư mục NEU-DET trên Drive!'

In [ ]:
# Cell 2: Clone/pull đúng nhánh duytung/14-neudet-dinov3-autoroi
import os, subprocess

REPO_URL = 'https://github.com/DuytungDeveloper/VB-CGCD-defect-testing-only.git'
BRANCH = 'duytung/14-neudet-dinov3-autoroi'
REPO_DIR = '/content/VB-CGCD-defect-testing-only'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)

subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--oneline'], check=True)

In [ ]:
# Cell 3: Cài đặt thư viện (nếu runtime mới)
!pip install -q peft transformers accelerate scipy scikit-learn datasets psutil prettytable
!python -c "import torch, jax, transformers, peft; print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available(), '| JAX:', jax.__version__, jax.devices())"

In [ ]:
# Cell 4: Chạy nền Bước A10 (NEU-DET) có auto-resume trên Google Drive
import os, subprocess
from google.colab import userdata

try:
    hf_tok = userdata.get('HF_TOKEN')
except Exception:
    hf_tok = os.environ.get('HF_TOKEN', '')

assert hf_tok, 'Vui lòng cấu hình HF_TOKEN trong Colab Secrets (biểu tượng chìa khóa bên trái)!'

env = os.environ.copy()
env['HF_TOKEN'] = hf_tok
env['HUGGING_FACE_HUB_TOKEN'] = hf_tok

log_path = '/content/drive/MyDrive/VB_CGCD_Work/neu_a10_dinov3_autoroi.log'
os.makedirs(os.path.dirname(log_path), exist_ok=True)
pid_path = '/tmp/neu_a10.pid'

with open(log_path, 'a') as lf:
    p = subprocess.Popen(
        ['bash', '/content/VB-CGCD-defect-testing-only/colab/run_neu_a10_bg.sh'],
        stdout=lf, stderr=subprocess.STDOUT, env=env, start_new_session=True
    )
with open(pid_path, 'w') as pf:
    pf.write(str(p.pid))
print(f'Đã khởi chạy Bước A10 (NEU-DET) chạy nền! PID={p.pid} | Log: {log_path}')

In [ ]:
# Cell 5: Kiểm tra tiến độ & xem bảng tổng hợp khi chạy xong
from pathlib import Path

pid = int(Path('/tmp/neu_a10.pid').read_text().strip()) if Path('/tmp/neu_a10.pid').exists() else -1
alive = False
if pid > 0 and Path(f'/proc/{pid}/status').exists():
    st = Path(f'/proc/{pid}/status').read_text()
    alive = ('State:\tZ' not in st and 'State: Z' not in st)
print(f'Trạng thái PID={pid}:', 'ĐANG CHẠY' if alive else 'ĐÃ DỪNG / XONG')

log_p = Path('/content/drive/MyDrive/VB_CGCD_Work/neu_a10_dinov3_autoroi.log')
if log_p.exists():
    lines = log_p.read_text(errors='replace').splitlines()
    print('\n--- 40 dòng log gần nhất ---')
    print('\n'.join(lines[-40:]))

sum_p = Path('/content/drive/MyDrive/VB_CGCD_Work/neu_a10_dinov3_autoroi_summary.txt')
if sum_p.exists() and not alive:
    print('\n--- BẢNG TỔNG HỢP BƯỚC A10 (NEU-DET) ---')
    print(sum_p.read_text())